# Part 2a: SP LR Sweep + Data Preprocessing

SVG Scaling Laws Project

**Runtime**: A100 GPU

**Steps**:
1. Setup (Drive + git pull)
2. Data preprocessing + tokenization (~15 min)
3. SP LR Sweep: Tiny × 5 LRs (~15 min)

**Run this notebook first** — it prepares the data used by all subsequent notebooks.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/svg-scaling-project
!git pull
!pip install -r requirements.txt -q

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Data Preprocessing + Tokenization

Download from HuggingFace, clean SVGs, train BPE tokenizer, convert to binary.

**Skip if `data/tokenized/train.bin` already exists.**

In [ ]:
import os
if os.path.exists('data/tokenized/train.bin'):
    print('data/tokenized/train.bin already exists, skipping preprocessing.')
    print('Delete data/tokenized/ and re-run this cell to force reprocessing.')
else:
    !python src/preprocess.py \
        --download starvector/svg-icons-simple \
        --output-dir data/processed \
        --min-len 50

    !python src/tokenize_data.py \
        --input-dir data/processed \
        --output-dir data/tokenized \
        --vocab-size 4096 \
        --max-token-len 2048

In [ ]:
# Verify: train tokens >= 100M
import json
with open('data/tokenized/tokenize_stats.json') as f:
    stats = json.load(f)
print(f"Train tokens: {stats['train']['total_tokens']:,}")
assert stats['train']['total_tokens'] >= 100_000_000, 'Need >= 100M train tokens!'

## 3. SP LR Sweep (Tiny × 5 LRs)

In [ ]:
import subprocess, time, json

lrs = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3']
results = []

for lr in lrs:
    output_dir = f'results/runs/lr_sweep/tiny_lr_{lr}'
    print(f'\n{"=" * 60}')
    print(f'SP LR Sweep: Tiny, LR={lr}')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', 'src/train.py',
        '--config', 'configs/tiny.yaml',
        '--learning-rate', lr,
        '--output-dir', output_dir,
        '--device', 'cuda',
    ])

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n\u2192 LR={lr}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        results.append({'lr': lr, **s})
        print(f'  final_val_loss={s["final_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

## 4. Results

In [ ]:
# Summary table (sorted by final_val_loss)
print(f'{"LR":>8s}  {"Final Val Loss":>14s}  {"Best Val Loss":>14s}  {"Final PPL":>10s}')
print(f'{"-"*8}  {"-"*14}  {"-"*14}  {"-"*10}')
for r in sorted(results, key=lambda x: x['final_val_loss']):
    print(f'{r["lr"]:>8s}  {r["final_val_loss"]:>14.4f}  {r["best_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

best = min(results, key=lambda x: x['final_val_loss'])
print(f'\n\u2192 Optimal SP LR: {best["lr"]} (final_val_loss={best["final_val_loss"]:.4f})')
print(f'  Use this LR for colab_scaling_study.ipynb')